# Impritng libraries

In [21]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


# Device

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## Define Model

In [23]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Linear(128, 1)
        self.bbox_head = nn.Linear(128, 4)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)

        obj_logit = self.classifier(x)
        bbox = torch.sigmoid(self.bbox_head(x))

        return obj_logit, bbox


## Load trained Weights

In [24]:
model = PneumoniaCNN().to(device)
model.load_state_dict(torch.load(r"D:\D drive\Project\AI That Explains Medical Images Like a Doctor\Another way\model creation\pneumonia_cnn.pth", map_location=device))
model.eval()


C:\Users\deven\AppData\Local\Temp\ipykernel_21592\2795692269.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"D:\D drive\Project\AI Tha

PneumoniaCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Linear(in_features=128, out_features=1, bias=True)
  (bbox_head): Linear(in_features=128, out_features=4, bias=True)
)

## Test Transform

In [25]:
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])


## Create Test Dataset

In [26]:
class XrayTestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.images = os.listdir(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        if self.transform:
            image = self.transform(image)

        return image, img_name, orig_w, orig_h


## DataLoader

In [27]:
test_dir = r"D:\D drive\Project\AI That Explains Medical Images Like a Doctor\Another way\Images Dataset\test"

test_dataset = XrayTestDataset(
    image_dir=test_dir,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,      # safe for GPU
    shuffle=False,
    num_workers=0
)

print("Total test images:", len(test_dataset))


Total test images: 3000


## Run Inference on ALL Test Images

In [28]:
results = []

threshold = 0.5  # pneumonia confidence threshold

with torch.no_grad():
    for images, img_names, orig_w, orig_h in test_loader:

        images = images.to(device)

        obj_logits, bboxes = model(images)
        probs = torch.sigmoid(obj_logits).squeeze(1)

        for i in range(len(img_names)):
            prob = probs[i].item()
            bbox = bboxes[i].cpu().numpy()

            if prob < threshold:
                results.append([
                    img_names[i],
                    prob,
                    0, 0, 0, 0,
                    "NO_PNEUMONIA"
                ])
            else:
                # Convert bbox from normalized → pixel
                x, y, w, h = bbox
                x *= orig_w[i]
                y *= orig_h[i]
                w *= orig_w[i]
                h *= orig_h[i]

                results.append([
                    img_names[i],
                    prob,
                    x, y, w, h,
                    "PNEUMONIA"
                ])


## Save Results to CSV

In [29]:
# df = pd.DataFrame(
#     results,
#     columns=[
#         "image_name",
#         "pneumonia_probability",
#         "x", "y", "width", "height",
#         "prediction"
#     ]
# )

# df.to_csv("test_predictions.csv", index=False)
# print("Saved test_predictions.csv")


In [30]:
import cv2
import json


In [31]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor):
        self.model.zero_grad()

        obj_logit, _ = self.model(input_tensor)

        # Backprop only classification output
        obj_logit.backward(torch.ones_like(obj_logit))

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)

        cam = torch.relu(cam)
        cam = cam.squeeze().detach().cpu().numpy()

        cam = cv2.resize(cam, (224, 224))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam


In [37]:
def detect_affected_lung(heatmap):
    h, w = heatmap.shape
    left = heatmap[:, :w//2].mean()
    right = heatmap[:, w//2:].mean()

    if abs(left - right) < 0.05:
        return "Both lungs"
    return "Right lung" if left > right else "Left lung"


def detect_opacity(heatmap, threshold=0.4):
    ratio = (heatmap > threshold).sum() / heatmap.size
    return bool(ratio > 0.05)


def detect_spread(heatmap, threshold=0.4):
    binary = np.uint8(heatmap > threshold)
    components, _ = cv2.connectedComponents(binary)
    return "Localized" if components <= 3 else "Diffuse"


In [38]:
def build_explanation_json(image_name, prob, bbox, heatmap):
    explanation = {
        "image_name": str(image_name),
        "prediction": "PNEUMONIA" if prob >= 0.5 else "NO_PNEUMONIA",
        "confidence": float(round(prob, 3)),

        # ✅ FIX HERE
        "opacity_detected": bool(detect_opacity(heatmap)),

        "affected_lung": detect_affected_lung(heatmap),
        "spread_type": detect_spread(heatmap),

        "bounding_box": {
            "x": float(bbox[0]),
            "y": float(bbox[1]),
            "width": float(bbox[2]),
            "height": float(bbox[3])
        },

        "visual_patterns": [],
        "disclaimer": "This AI result is for educational purposes only and not a medical diagnosis."
    }

    if explanation["opacity_detected"]:
        explanation["visual_patterns"].extend([
            "increased opacity",
            "loss of normal lung texture"
        ])

    explanation["visual_patterns"].append(
        "localized lung involvement" if explanation["spread_type"] == "Localized"
        else "diffuse lung involvement"
    )

    return explanation


In [39]:
gradcam = GradCAM(model, model.features[6])

all_results = []

for images, img_names, orig_w, orig_h in test_loader:

    images = images.to(device)

    obj_logits, bboxes = model(images)
    probs = torch.sigmoid(obj_logits).squeeze(1)

    for i in range(len(img_names)):
        image_tensor = images[i].unsqueeze(0)
        prob = probs[i].item()
        bbox = bboxes[i].detach().cpu().numpy()

        heatmap = gradcam.generate(image_tensor)

        explanation_json = build_explanation_json(
            img_names[i],
            prob,
            bbox,
            heatmap
        )

        all_results.append(explanation_json)


c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1827: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


In [40]:
output_path = "pneumonia_explanations.json"

with open(output_path, "w") as f:
    json.dump(all_results, f, indent=4)

print("Saved explanations to:", output_path)


Saved explanations to: pneumonia_explanations.json
